# Lab 8.8 &mdash; Critique & Debate

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 4 &middot; Module 8 &mdash; Multi-Agent Collaboration &amp; Decision Making**

### What you'll do
- Run a propose -> critique -> revise loop
- Let an independent critic gate on quality
- Cap the loop so two agents can't argue forever

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`). The **graded** cells assert only on the deterministic parts you build &mdash; routing, synthesis, tool wiring, agent structure &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable. You are building the **multi-agent customer-service chatbot** &mdash; the client's Lab 4.2.

**Reference:** [Module 8 slides &mdash; Critique & debate](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 8 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-08-08"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
Instead of averaging, **stress-test** one answer (deck slide 13): one agent **proposes**, an
independent **critic** tries to find what's wrong, the proposer **revises** &mdash; repeat until the
critic approves or a **cap** is hit. Generating and evaluating are **different skills**, so a separate
skeptic catches the author's blind spots (just like code review). Always **cap** the loop.

In [ ]:
# proposer improves its draft each round; critic approves only a GROUNDED answer.
def proposer(prev):
    if prev is None: return "draft v1"
    if "v1" in prev: return "draft v2 grounded"
    return prev
def critic(answer):
    return "approve" if "grounded" in answer else "revise"
def critic_never(answer):
    return "revise"
print("proposer & critics ready")

## Your Turn
Complete `critique_loop`: get the critic's verdict and stop when it approves (or at the cap).

In [ ]:
def critique_loop(proposer, critic, max_rounds=3):
    answer = None
    for r in range(max_rounds):
        answer = proposer(answer)
        verdict = ___   # TODO: ask the critic to review this answer
        if ___:   # TODO: the critic approved
            return {"answer": answer, "rounds": r + 1, "approved": True}
    return {"answer": answer, "rounds": max_rounds, "approved": False}

try:
    print('approved:', critique_loop(proposer, critic))
    print('capped  :', critique_loop(proposer, critic_never))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("the loop returns an approved answer", lambda: critique_loop(proposer, critic)["approved"] is True)
expect_true("it approves in the expected round (2)", lambda: critique_loop(proposer, critic)["rounds"] == 2)
expect_true("a never-satisfied critic hits the cap", lambda: critique_loop(proposer, critic_never, max_rounds=3)["rounds"] == 3)
expect_true("a capped run is marked not approved", lambda: critique_loop(proposer, critic_never)["approved"] is False)
expect_true("the critic actually gates the result", lambda: "grounded" in critique_loop(proposer, critic)["answer"])

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
A separate critic raises quality because evaluating is a different skill from generating -- and the cap keeps the debate from running forever. Use it when being right beats being fast.

[&#8592; All Module 8 labs](./index.html) &nbsp;&middot;&nbsp; [Module 8 slides](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>